# Multi-Horizon Probabilistic Framework for Indian Electricity Market Intelligence
## Notebook 1: Data Loading & Preparation

This notebook covers:
1. **Dataset #1**: IEX Day-Ahead Market (DAM) prices & volumes — from Kaggle
2. **Dataset #2**: India hourly electricity demand + solar + wind generation — from Mendeley Data
3. **Dataset #3**: State-level daily electricity demand with weather covariates — from Zenodo
4. **Dataset #4**: Calibrated synthetic RTM dataset (engineered to study DAM→RTM stress propagation)
5. **Unified merge** of all four into a single analysis-ready DataFrame

---

### Why these 4 datasets?

| # | Dataset | What it gives us | Role in project |
|---|---------|------------------|-----------------|
| 1 | IEX DAM (Kaggle) | Real MCP, MCV, Purchase/Sell bids, 15-min blocks | **Core predictor** — DAM bidding signals |
| 2 | Demand + Solar + Wind (Mendeley) | Hourly grid demand, renewable generation | **Exogenous features** — duck curve, ramp rates |
| 3 | State demand + weather (Zenodo) | Daily peak demand, temperature, wind | **Seasonal & weather context** for stress modelling |
| 4 | Synthetic RTM (generated) | RTM prices, volumes with embedded DAM coupling | **Target variable** — RTM stress events to predict |

> **Key insight from published research**: The best electricity price forecasting papers (European EPEX, Nord Pool, PJM) all combine *market clearing data* + *renewable generation* + *demand/load* + *weather*. No single dataset alone is sufficient. Our uniqueness is the **cross-market DAM→RTM stress propagation** angle — something not studied in prior Indian market research.

---
## Step 0: Install Dependencies & Setup

In [ ]:
# Run once in Colab
!pip install -q kaggle opendatasets openpyxl xlrd

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project directory in Drive
PROJECT_DIR = '/content/drive/MyDrive/Indian_Electricity_Market_Intelligence'
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Project directory: {PROJECT_DIR}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

---
## Dataset #1: IEX Day-Ahead Market (DAM) — Real Market Data from Kaggle

**Source**: [Indian Energy Exchange - Time Series Forecasting](https://www.kaggle.com/datasets/mukund604/indian-energy-exchange-time-series-forecasting)  
**What it contains**: DAM Market Clearing Prices (MCP in Rs/MWh) from 2019–2024, 15-minute time blocks  
**License**: Apache 2.0  

### How to download:

**Option A — Using Kaggle API (recommended):**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → scroll to "API" → click "Create New Token"
2. This downloads a `kaggle.json` file
3. Upload it when prompted below

**Option B — Manual download:**
1. Visit the Kaggle dataset page linked above
2. Click "Download" → save the ZIP
3. Upload the CSV to your Google Drive under `Indian_Electricity_Market_Intelligence/data/raw/`

In [ ]:
# === OPTION A: Kaggle API Download ===
# Upload your kaggle.json when prompted

from google.colab import files

# Check if kaggle.json already exists
kaggle_dir = os.path.expanduser('~/.kaggle')
kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')

if not os.path.exists(kaggle_json):
    print("Upload your kaggle.json file (from kaggle.com/settings -> API -> Create New Token):")
    uploaded = files.upload()
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(kaggle_json, 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(kaggle_json, 0o600)
    print("✅ kaggle.json configured!")
else:
    print("✅ kaggle.json already configured.")

In [ ]:
# Download the IEX DAM dataset from Kaggle
KAGGLE_DAM_DATASET = 'mukund604/indian-energy-exchange-time-series-forecasting'
DAM_DOWNLOAD_DIR = os.path.join(RAW_DIR, 'kaggle_iex_dam')
os.makedirs(DAM_DOWNLOAD_DIR, exist_ok=True)

!kaggle datasets download -d {KAGGLE_DAM_DATASET} -p {DAM_DOWNLOAD_DIR} --unzip --force

# List downloaded files
dam_files = os.listdir(DAM_DOWNLOAD_DIR)
print(f"\nDownloaded files: {dam_files}")

In [ ]:
# Load the DAM dataset — handles both CSV and Excel formats
def load_dam_from_directory(directory):
    """Load all CSV/Excel files from the Kaggle download directory into one DataFrame."""
    frames = []
    for fname in sorted(os.listdir(directory)):
        fpath = os.path.join(directory, fname)
        if fname.endswith('.csv'):
            frames.append(pd.read_csv(fpath))
        elif fname.endswith(('.xlsx', '.xls')):
            frames.append(pd.read_excel(fpath))
    if not frames:
        raise FileNotFoundError(f"No CSV/Excel files found in {directory}")
    df = pd.concat(frames, ignore_index=True)
    return df

dam = load_dam_from_directory(DAM_DOWNLOAD_DIR)
print(f"DAM dataset loaded: {dam.shape[0]:,} rows × {dam.shape[1]} columns")
print(f"\nColumns: {list(dam.columns)}")
print(f"\nDate range: {dam.iloc[:, 0].min()} to {dam.iloc[:, 0].max()}")
dam.head()

In [ ]:
# Standardize DAM column names to our canonical schema
def standardize_dam_columns(df):
    """Map various IEX DAM column naming conventions to a standard schema."""
    df_out = df.copy()
    rename_map = {}

    for col in df_out.columns:
        cl = col.strip().lower().replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')

        if 'date' in cl and 'trade' not in cl:
            rename_map[col] = 'date'
        elif cl in ('hour', 'hr'):
            rename_map[col] = 'hour'
        elif 'time_block' in cl or 'block_desc' in cl or 'time_slot' in cl:
            rename_map[col] = 'time_block'
        elif cl in ('block_no', 'block', 'token', 'block_number'):
            rename_map[col] = 'block_no'
        elif 'purchase' in cl and ('bid' in cl or 'mw' in cl):
            rename_map[col] = 'dam_purchase_bid_mw'
        elif 'sell' in cl and ('bid' in cl or 'mw' in cl):
            rename_map[col] = 'dam_sell_bid_mw'
        elif 'mcv' in cl or 'cleared_vol' in cl or 'traded_vol' in cl or 'volume' in cl:
            rename_map[col] = 'dam_mcv_mw'
        elif 'mcp' in cl or 'clearing_price' in cl or ('price' in cl and 'spread' not in cl):
            rename_map[col] = 'dam_mcp'

    df_out.rename(columns=rename_map, inplace=True)

    # Parse date
    if 'date' in df_out.columns:
        df_out['date'] = pd.to_datetime(df_out['date'], dayfirst=True, errors='coerce')

    # Create block_no if missing (infer from row order within each date)
    if 'block_no' not in df_out.columns:
        df_out['block_no'] = df_out.groupby('date').cumcount() + 1

    # Create hour if missing
    if 'hour' not in df_out.columns:
        df_out['hour'] = ((df_out['block_no'] - 1) // 4) + 1

    return df_out

dam = standardize_dam_columns(dam)
print(f"Standardized DAM columns: {list(dam.columns)}")
print(f"Date range: {dam['date'].min().date()} to {dam['date'].max().date()}")
print(f"Blocks per day (sample): {dam.groupby('date')['block_no'].nunique().mode().values}")
dam.head()

---
## Dataset #2: India Electricity Demand + Solar + Wind Generation (Mendeley Data)

**Source**: [Mendeley Data — DOI: 10.17632/y58jknpgs8.2](https://doi.org/10.17632/y58jknpgs8.2)  
**Authors**: Mukherjee, Kalita & Kumar (IIT Guwahati & PGCIL)  
**What it contains**: Hourly all-India electricity demand, solar generation, and wind generation (Sept 2021 – June 2025) from Grid-India NERLDC  

### How to download:
1. Visit the DOI link above
2. Click "Download All" or download individual files
3. Upload the Excel/CSV files to Google Drive under `Indian_Electricity_Market_Intelligence/data/raw/mendeley_demand/`

> **Why this dataset matters**: Solar and wind generation directly cause the "duck curve" — midday price depression followed by the sharp evening ramp that drives market stress. Without renewable generation data, we cannot model the physical drivers of DAM→RTM price divergence.

In [ ]:
# === Loading Mendeley demand + renewables data ===
# After downloading from the DOI link, place files in this folder:
MENDELEY_DIR = os.path.join(RAW_DIR, 'mendeley_demand')
os.makedirs(MENDELEY_DIR, exist_ok=True)

# Option: Direct upload to Colab session (if not using Drive)
# from google.colab import files
# uploaded = files.upload()  # Select the downloaded Excel/CSV files

def load_mendeley_demand(directory):
    """Load Mendeley hourly demand, solar, and wind generation data."""
    frames = []
    for fname in sorted(os.listdir(directory)):
        fpath = os.path.join(directory, fname)
        try:
            if fname.endswith('.csv'):
                frames.append(pd.read_csv(fpath))
            elif fname.endswith(('.xlsx', '.xls')):
                # Some Mendeley files have multiple sheets
                xls = pd.ExcelFile(fpath)
                for sheet in xls.sheet_names:
                    frames.append(pd.read_excel(fpath, sheet_name=sheet))
        except Exception as e:
            print(f"  Warning: Could not read {fname}: {e}")

    if not frames:
        print(f"⚠️  No files found in {directory}.")
        print(f"   Download from: https://doi.org/10.17632/y58jknpgs8.2")
        print(f"   Place files in: {directory}")
        return None

    df = pd.concat(frames, ignore_index=True)
    return df

demand_renewable = load_mendeley_demand(MENDELEY_DIR)

if demand_renewable is not None:
    print(f"Demand+Renewables dataset loaded: {demand_renewable.shape[0]:,} rows × {demand_renewable.shape[1]} columns")
    print(f"Columns: {list(demand_renewable.columns)}")
    display(demand_renewable.head())
else:
    print("\n📥 Dataset #2 not yet downloaded. Follow the instructions above.")
    print("   This is optional — the project works without it but is stronger with it.")

---
## Dataset #3: State-Level Daily Demand + Weather Covariates (Zenodo)

**Source**: [Zenodo — DOI: 10.5281/zenodo.14983362](https://doi.org/10.5281/zenodo.14983362)  
**Authors**: Kieran Hunt & Hannah Bloomfield (University of Reading)  
**What it contains**: Daily electricity demand for all Indian states (2014–2024) with temperature, wind speed, solar radiation, cloud cover  

### How to download:
1. Visit the DOI link above
2. Download the ZIP archive
3. Extract and upload CSV files to Google Drive under `Indian_Electricity_Market_Intelligence/data/raw/zenodo_state_demand/`

> **Why this dataset matters**: Weather (especially temperature) is the single biggest exogenous driver of electricity demand in India. Summer heatwaves push demand to record highs, causing DAM prices to hit CERC caps (₹10/kWh). Including temperature lets us separate weather-driven stress from pure market-structure stress.

In [ ]:
# === Loading Zenodo state-level demand + weather data ===
ZENODO_DIR = os.path.join(RAW_DIR, 'zenodo_state_demand')
os.makedirs(ZENODO_DIR, exist_ok=True)

def load_zenodo_state_demand(directory):
    """Load state-level daily demand CSVs from the Zenodo Hunt & Bloomfield dataset."""
    frames = []
    for fname in sorted(os.listdir(directory)):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(directory, fname)
        try:
            df_state = pd.read_csv(fpath)
            # Add state name from filename
            state_name = fname.replace('.csv', '').replace('_', ' ').title()
            df_state['state'] = state_name
            frames.append(df_state)
        except Exception as e:
            print(f"  Warning: Could not read {fname}: {e}")

    if not frames:
        print(f"⚠️  No files found in {directory}.")
        print(f"   Download from: https://doi.org/10.5281/zenodo.14983362")
        print(f"   Place CSVs in: {directory}")
        return None

    df = pd.concat(frames, ignore_index=True)
    return df

state_demand = load_zenodo_state_demand(ZENODO_DIR)

if state_demand is not None:
    print(f"State demand dataset loaded: {state_demand.shape[0]:,} rows × {state_demand.shape[1]} columns")
    print(f"States: {state_demand['state'].nunique()}")
    print(f"Columns: {list(state_demand.columns)}")
    display(state_demand.head())
else:
    print("\n📥 Dataset #3 not yet downloaded. Follow the instructions above.")
    print("   This is optional — adds weather context for seasonal stress modelling.")

---
## Dataset #4: Calibrated Synthetic RTM Dataset

### Why do we need this?

The IEX **Real-Time Market (RTM)** launched in June 2020. Unlike the DAM, there is **no single clean public Kaggle/Mendeley dump** of RTM block-wise data. Downloading RTM data requires either:
- Manual 30-day batch exports from the IEX portal (tedious for multi-year coverage)
- Selenium/Playwright browser automation (blocked by Cloudflare)
- IEX Member API access (requires institutional registration + API token)

Our synthetic RTM generator solves this by creating data that:
1. **Matches real IEX RTM schema exactly** — same columns, same 96-block structure, same price bounds
2. **Embeds the DAM→RTM coupling mechanism** — so we can test our central research hypothesis
3. **Is schema-compatible** — when you later get real RTM data, swap it in with zero code changes
4. **Captures Indian market physics** — duck curve, evening ramp stress, seasonal heatwaves, monsoon dips

### What makes this different from random synthetic data?

The generator uses **real electricity market mechanics**:
- Price is a **non-linear function of market tightness** (purchase/sell ratio), not random noise
- Evening ramp stress (blocks 68–82) probabilistically triggers RTM spikes via **logistic coupling**
- All prices are bounded by CERC regulatory caps (₹500–₹10,000/MWh)
- Seasonality follows the Indian power calendar (April–June heatwave, July–Sept monsoon)

In [ ]:
# ==============================================================================
# DATASET #4: CALIBRATED SYNTHETIC RTM GENERATOR
# ==============================================================================

def generate_synthetic_rtm(dam_df: pd.DataFrame, random_seed: int = 42) -> pd.DataFrame:
    """
    Generate a synthetic Real-Time Market (RTM) dataset that is:
    - Temporally aligned with the real DAM data (same dates, same blocks)
    - Coupled to DAM bidding signals (tightness → RTM stress propagation)
    - Bounded by CERC regulatory price limits
    - Schema-identical to real IEX RTM exports

    The coupling mechanism:
    When DAM purchase-to-sell ratio is high (tight market), the probability
    of a real-time stress event increases via a logistic function. This is
    the exact hypothesis we want to test — and we embed it at known strength
    so we can verify our statistical tests detect it correctly.
    """
    np.random.seed(random_seed)
    df = dam_df.copy()
    n = len(df)

    # Ensure we have the DAM columns we need
    required = ['date', 'block_no', 'dam_mcp']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"DAM DataFrame missing required columns: {missing}")

    blocks = df['block_no'].values

    # --- Step 1: Compute DAM market tightness ---
    if 'dam_purchase_bid_mw' in df.columns and 'dam_sell_bid_mw' in df.columns:
        dam_ratio = df['dam_purchase_bid_mw'].values / np.maximum(df['dam_sell_bid_mw'].values, 1.0)
    else:
        # If DAM dataset only has MCP (no bid volumes), infer tightness from price percentile
        price_pctile = df['dam_mcp'].rank(pct=True).values
        dam_ratio = 0.7 + 0.6 * price_pctile  # Maps 0-100th percentile to 0.7-1.3 ratio

    # --- Step 2: RTM stress probability (logistic coupling to DAM tightness) ---
    # Core coupling: P(stress) = sigmoid(3.5 * (ratio - 1.12))
    stress_prob = 1.0 / (1.0 + np.exp(-3.5 * (dam_ratio - 1.12)))

    # Evening ramp amplification (blocks 68-82, i.e. 17:00-20:30 IST)
    evening_mask = (blocks >= 68) & (blocks <= 82)
    stress_prob[evening_mask] = np.clip(stress_prob[evening_mask] * 1.6, 0.0, 0.95)

    # Morning ramp (blocks 28-36, i.e. 07:00-09:00) — mild amplification
    morning_mask = (blocks >= 28) & (blocks <= 36)
    stress_prob[morning_mask] = np.clip(stress_prob[morning_mask] * 1.15, 0.0, 0.85)

    # Night damping (blocks 1-16, i.e. 00:00-04:00) — lower stress
    night_mask = blocks <= 16
    stress_prob[night_mask] = stress_prob[night_mask] * 0.5

    # --- Step 3: Generate RTM shock events ---
    shock_active = (np.random.uniform(0, 1, n) < stress_prob).astype(float)

    # --- Step 4: RTM Purchase and Sell Bids ---
    if 'dam_purchase_bid_mw' in df.columns:
        base_rtm_purchase = df['dam_purchase_bid_mw'].values * np.random.uniform(0.20, 0.35, n)
        base_rtm_sell = df['dam_sell_bid_mw'].values * np.random.uniform(0.18, 0.32, n)
    else:
        # Estimate from price if volumes not available
        base_rtm_purchase = 2000 + 800 * dam_ratio + np.random.normal(0, 200, n)
        base_rtm_sell = 2500 + np.random.normal(0, 300, n)

    rtm_purchase = np.maximum(500, base_rtm_purchase + shock_active * np.random.uniform(1500, 3500, n))
    rtm_sell = np.maximum(400, base_rtm_sell - shock_active * np.random.uniform(500, 1500, n))

    rtm_ratio = rtm_purchase / np.maximum(rtm_sell, 1.0)
    rtm_mcv = np.minimum(rtm_purchase, rtm_sell) * np.random.uniform(0.85, 0.95, n)

    # --- Step 5: RTM Market Clearing Price ---
    dam_price = df['dam_mcp'].values.astype(float)
    spread_multiplier = (rtm_ratio / np.maximum(dam_ratio, 0.5)) ** 1.6
    rtm_mcp = dam_price * spread_multiplier + shock_active * np.random.uniform(1200, 4000, n)
    # Add baseline noise
    rtm_mcp += np.random.normal(0, 120, n)
    # Enforce CERC price bounds: floor Rs 0/MWh, ceiling Rs 10,000/MWh
    rtm_mcp = np.clip(rtm_mcp, 500.0, 10000.0)

    # --- Step 6: RTM Session ID (48 sessions, each covering 2 blocks) ---
    session_id = ((blocks - 1) // 2) + 1

    # --- Step 7: Assemble RTM DataFrame ---
    rtm = pd.DataFrame({
        'date': df['date'].values,
        'block_no': blocks,
        'hour': df['hour'].values if 'hour' in df.columns else ((blocks - 1) // 4) + 1,
        'session_id': session_id,
        'rtm_purchase_bid_mw': np.round(rtm_purchase, 2),
        'rtm_sell_bid_mw': np.round(rtm_sell, 2),
        'rtm_mcv_mw': np.round(rtm_mcv, 2),
        'rtm_mcp': np.round(rtm_mcp, 2),
        '_rtm_shock_flag': shock_active.astype(int)  # Ground truth for verification
    })

    return rtm

# Generate RTM aligned to our real DAM dates and blocks
rtm = generate_synthetic_rtm(dam)
print(f"Synthetic RTM generated: {rtm.shape[0]:,} rows × {rtm.shape[1]} columns")
print(f"Date range: {rtm['date'].min().date()} to {rtm['date'].max().date()}")
print(f"RTM shock rate: {rtm['_rtm_shock_flag'].mean()*100:.1f}%")
rtm.head()

---
## Step 5: Merge DAM + RTM into Unified Market Dataset

In [ ]:
# ==============================================================================
# UNIFIED MARKET DATASET: DAM + RTM aligned on (date, block_no)
# ==============================================================================

df_market = pd.merge(dam, rtm, on=['date', 'block_no'], how='inner', suffixes=('', '_rtm_dup'))

# Clean up duplicate columns from merge
dup_cols = [c for c in df_market.columns if c.endswith('_rtm_dup')]
df_market.drop(columns=dup_cols, inplace=True)

# Use 'hour' from DAM if both exist
if 'hour_x' in df_market.columns:
    df_market.rename(columns={'hour_x': 'hour'}, inplace=True)
    df_market.drop(columns=[c for c in df_market.columns if c == 'hour_y'], errors='ignore', inplace=True)

# --- Derived features for analysis ---
# DAM market tightness
if 'dam_purchase_bid_mw' in df_market.columns and 'dam_sell_bid_mw' in df_market.columns:
    df_market['dam_imbalance_ratio'] = (
        df_market['dam_purchase_bid_mw'] / np.maximum(df_market['dam_sell_bid_mw'], 1.0)
    )
else:
    # Proxy from price percentile
    df_market['dam_imbalance_ratio'] = 0.7 + 0.6 * df_market['dam_mcp'].rank(pct=True)

# RTM market tightness
df_market['rtm_imbalance_ratio'] = (
    df_market['rtm_purchase_bid_mw'] / np.maximum(df_market['rtm_sell_bid_mw'], 1.0)
)

# Cross-market spread (our key research variable)
df_market['dam_rtm_spread'] = df_market['rtm_mcp'] - df_market['dam_mcp']

# Target: RTM price spike (>= P90 of RTM prices)
rtm_p90 = df_market['rtm_mcp'].quantile(0.90)
df_market['rtm_spike'] = (df_market['rtm_mcp'] >= rtm_p90).astype(int)

# Time-of-day regime classification
def classify_regime(b):
    if b <= 24:   return 'Night (00:00-06:00)'
    elif b <= 40: return 'Morning (06:00-10:00)'
    elif b <= 64: return 'Solar Peak (10:00-16:00)'
    elif b <= 84: return 'Evening Ramp (16:00-21:00)'
    else:         return 'Late (21:00-24:00)'

df_market['regime'] = df_market['block_no'].apply(classify_regime)

print(f"\n✅ Unified market dataset: {df_market.shape[0]:,} rows × {df_market.shape[1]} columns")
print(f"Date range: {df_market['date'].min().date()} to {df_market['date'].max().date()}")
print(f"RTM P90 threshold: ₹{rtm_p90:,.0f}/MWh (₹{rtm_p90/1000:.2f}/kWh)")
print(f"RTM spike rate: {df_market['rtm_spike'].mean()*100:.1f}%")
print(f"\nAll columns: {list(df_market.columns)}")
df_market.head()

---
## Step 6: Enrich with Demand & Weather Data (if available)

In [ ]:
# ==============================================================================
# OPTIONAL ENRICHMENT: Merge in demand/renewables and weather data
# ==============================================================================

# --- Mendeley Demand + Renewables (hourly → mapped to blocks) ---
if demand_renewable is not None:
    print("Merging Mendeley demand + renewables data...")
    # Standardize date and hour columns (adjust based on actual column names)
    dr = demand_renewable.copy()
    # Attempt to find date and hour columns
    date_cols = [c for c in dr.columns if 'date' in c.lower()]
    hour_cols = [c for c in dr.columns if 'hour' in c.lower() or 'time' in c.lower()]
    if date_cols and hour_cols:
        dr.rename(columns={date_cols[0]: 'date', hour_cols[0]: 'hour'}, inplace=True)
        dr['date'] = pd.to_datetime(dr['date'], errors='coerce')
        df_market = pd.merge(df_market, dr, on=['date', 'hour'], how='left')
        print(f"  ✅ Merged. New shape: {df_market.shape}")
    else:
        print(f"  ⚠️ Could not auto-detect date/hour columns: {list(dr.columns)}")
        print(f"  You may need to manually map columns.")
else:
    print("ℹ️  Mendeley data not loaded — skipping demand/renewables enrichment.")
    print("   Project proceeds with market-only features (DAM + RTM).")

# --- Zenodo State Demand + Weather (daily) ---
if state_demand is not None:
    print("\nAggregating Zenodo state-level daily data to all-India level...")
    sd = state_demand.copy()
    date_cols = [c for c in sd.columns if 'date' in c.lower()]
    if date_cols:
        sd.rename(columns={date_cols[0]: 'date'}, inplace=True)
        sd['date'] = pd.to_datetime(sd['date'], errors='coerce')

        # Aggregate to all-India daily level
        numeric_cols = sd.select_dtypes(include=[np.number]).columns.tolist()
        daily_agg = sd.groupby('date')[numeric_cols].mean().reset_index()
        daily_agg.columns = ['date'] + [f'daily_{c}' for c in numeric_cols]

        df_market = pd.merge(df_market, daily_agg, on='date', how='left')
        print(f"  ✅ Merged. New shape: {df_market.shape}")
    else:
        print(f"  ⚠️ Could not find date column in Zenodo data.")
else:
    print("ℹ️  Zenodo data not loaded — skipping weather enrichment.")
    print("   Project proceeds without temperature/weather covariates.")

---
## Step 7: Data Quality Report & Sanity Checks

In [ ]:
# ==============================================================================
# DATA QUALITY AUDIT
# ==============================================================================

print("=" * 65)
print("DATA QUALITY REPORT")
print("=" * 65)

# 1. Basic shape and coverage
n_days = df_market['date'].nunique()
n_blocks = len(df_market)
expected_blocks = n_days * 96
print(f"\n📊 Coverage:")
print(f"   Days: {n_days:,}")
print(f"   Total blocks: {n_blocks:,}")
print(f"   Expected (96/day): {expected_blocks:,}")
print(f"   Completeness: {n_blocks/expected_blocks*100:.1f}%")

# 2. Missing values
core_cols = ['dam_mcp', 'rtm_mcp', 'dam_rtm_spread', 'dam_imbalance_ratio']
core_cols = [c for c in core_cols if c in df_market.columns]
print(f"\n🔍 Missing values (core columns):")
for col in core_cols:
    miss = df_market[col].isna().sum()
    print(f"   {col}: {miss} ({miss/n_blocks*100:.2f}%)")

# 3. Price range validation
print(f"\n💰 Price ranges (Rs/MWh):")
for col in ['dam_mcp', 'rtm_mcp']:
    if col in df_market.columns:
        vals = df_market[col].dropna()
        print(f"   {col}: min={vals.min():.0f}, median={vals.median():.0f}, "
              f"mean={vals.mean():.0f}, P90={vals.quantile(0.9):.0f}, max={vals.max():.0f}")

# 4. Block integrity check
blocks_per_day = df_market.groupby('date')['block_no'].nunique()
incomplete_days = blocks_per_day[blocks_per_day < 96]
print(f"\n📅 Block integrity:")
print(f"   Days with full 96 blocks: {(blocks_per_day == 96).sum()} / {n_days}")
if len(incomplete_days) > 0:
    print(f"   Incomplete days: {len(incomplete_days)} (min blocks: {incomplete_days.min()})")

# 5. Cross-market spread statistics
if 'dam_rtm_spread' in df_market.columns:
    spread = df_market['dam_rtm_spread'].dropna()
    print(f"\n📈 DAM→RTM Spread (Rs/MWh):")
    print(f"   Mean: {spread.mean():.0f} | Std: {spread.std():.0f}")
    print(f"   P10: {spread.quantile(0.1):.0f} | P90: {spread.quantile(0.9):.0f}")
    print(f"   RTM > DAM in {(spread > 0).mean()*100:.1f}% of blocks")

In [ ]:
# ==============================================================================
# VISUAL SANITY CHECK: 4-Panel Diagnostic
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Diurnal price profile (DAM vs RTM)
diurnal = df_market.groupby('block_no')[['dam_mcp', 'rtm_mcp']].mean() / 1000
axes[0, 0].plot(diurnal.index, diurnal['dam_mcp'], label='DAM MCP', color='#2563eb', lw=2.5)
axes[0, 0].plot(diurnal.index, diurnal['rtm_mcp'], label='RTM MCP', color='#dc2626', lw=2, ls='--')
axes[0, 0].axvspan(68, 82, color='#fef08a', alpha=0.4, label='Evening Ramp (68-82)')
axes[0, 0].set_title('1. Average Diurnal Price Profile', fontweight='bold')
axes[0, 0].set_ylabel('Price (₹/kWh)')
axes[0, 0].legend(loc='upper left', fontsize=9)
axes[0, 0].grid(True, ls=':', alpha=0.5)

# 2. DAM-RTM spread by time-of-day
spread_diurnal = df_market.groupby('block_no')['dam_rtm_spread'].mean() / 1000
axes[0, 1].fill_between(spread_diurnal.index, 0, spread_diurnal.values,
                        where=spread_diurnal.values > 0, color='#ef4444', alpha=0.4, label='RTM > DAM')
axes[0, 1].fill_between(spread_diurnal.index, 0, spread_diurnal.values,
                        where=spread_diurnal.values <= 0, color='#22c55e', alpha=0.4, label='DAM > RTM')
axes[0, 1].axhline(0, color='black', lw=0.8)
axes[0, 1].axvspan(68, 82, color='#fef08a', alpha=0.3)
axes[0, 1].set_title('2. Mean DAM→RTM Price Spread', fontweight='bold')
axes[0, 1].set_ylabel('RTM − DAM (₹/kWh)')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, ls=':', alpha=0.5)

# 3. RTM spike rate by regime
spike_by_regime = df_market.groupby('regime')['rtm_spike'].mean() * 100
spike_by_regime = spike_by_regime.reindex(['Night (00:00-06:00)', 'Morning (06:00-10:00)',
                                          'Solar Peak (10:00-16:00)', 'Evening Ramp (16:00-21:00)',
                                          'Late (21:00-24:00)'])
colors = ['#94a3b8', '#60a5fa', '#4ade80', '#f87171', '#c084fc']
axes[1, 0].bar(range(len(spike_by_regime)), spike_by_regime.values, color=colors, width=0.6)
axes[1, 0].set_xticks(range(len(spike_by_regime)))
axes[1, 0].set_xticklabels(['Night', 'Morning', 'Solar Peak', 'Evening Ramp', 'Late'], fontsize=9)
axes[1, 0].set_title('3. RTM Spike Rate by Grid Regime', fontweight='bold')
axes[1, 0].set_ylabel('Spike Rate (%)')
for i, v in enumerate(spike_by_regime.values):
    axes[1, 0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
axes[1, 0].grid(True, ls=':', alpha=0.5, axis='y')

# 4. Price distribution comparison
axes[1, 1].hist(df_market['dam_mcp'] / 1000, bins=60, alpha=0.6, color='#2563eb',
                label='DAM', density=True)
axes[1, 1].hist(df_market['rtm_mcp'] / 1000, bins=60, alpha=0.5, color='#dc2626',
                label='RTM', density=True)
axes[1, 1].axvline(rtm_p90 / 1000, color='red', ls='--', lw=1.5, label=f'RTM P90 = ₹{rtm_p90/1000:.1f}/kWh')
axes[1, 1].set_title('4. Price Distribution: DAM vs RTM', fontweight='bold')
axes[1, 1].set_xlabel('Price (₹/kWh)')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, ls=':', alpha=0.5)

for ax in axes.flat:
    ax.set_xlabel(ax.get_xlabel() or '15-Min Block (1-96)', fontsize=9)

plt.suptitle('Indian Electricity Market: DAM + RTM Data Quality & Structure',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 8: Save Processed Data to Google Drive

In [ ]:
# ==============================================================================
# SAVE ALL PROCESSED DATASETS
# ==============================================================================

# 1. Save individual market DataFrames
dam.to_csv(os.path.join(PROCESSED_DIR, 'dam_standardized.csv'), index=False)
rtm.to_csv(os.path.join(PROCESSED_DIR, 'rtm_synthetic.csv'), index=False)

# 2. Save the unified market dataset (this is what all subsequent notebooks use)
df_market.to_csv(os.path.join(PROCESSED_DIR, 'unified_market_dataset.csv'), index=False)

# 3. Save metadata
metadata = {
    'created': pd.Timestamp.now().isoformat(),
    'dam_source': 'Kaggle - mukund604/indian-energy-exchange-time-series-forecasting',
    'rtm_source': 'Synthetic (calibrated to DAM, schema-compatible with real IEX RTM)',
    'demand_source': 'Mendeley DOI:10.17632/y58jknpgs8.2 (if loaded)',
    'weather_source': 'Zenodo DOI:10.5281/zenodo.14983362 (if loaded)',
    'total_blocks': len(df_market),
    'date_range': f"{df_market['date'].min().date()} to {df_market['date'].max().date()}",
    'rtm_p90_threshold': float(rtm_p90),
    'columns': list(df_market.columns)
}
pd.Series(metadata).to_json(os.path.join(PROCESSED_DIR, 'dataset_metadata.json'), indent=2)

print("✅ All datasets saved to Google Drive:")
for f in sorted(os.listdir(PROCESSED_DIR)):
    fsize = os.path.getsize(os.path.join(PROCESSED_DIR, f)) / (1024 * 1024)
    print(f"   📄 {f} ({fsize:.1f} MB)")

print(f"\n📁 Location: {PROCESSED_DIR}")
print(f"\n🔑 Key variable for next notebook:")
print(f"   df_market — {df_market.shape[0]:,} rows × {df_market.shape[1]} columns")

---
## Summary: What We Have Now

| DataFrame | Source | Rows | Key Columns | Status |
|-----------|--------|------|-------------|--------|
| `dam` | Kaggle (real IEX DAM) | Varies | `dam_mcp`, `dam_purchase_bid_mw`, `dam_sell_bid_mw`, `dam_mcv_mw` | ✅ Real data |
| `rtm` | Synthetic (calibrated) | Same as DAM | `rtm_mcp`, `rtm_purchase_bid_mw`, `rtm_sell_bid_mw`, `rtm_mcv_mw` | ✅ Generated |
| `demand_renewable` | Mendeley (real) | Hourly | Demand, Solar MW, Wind MW | 📥 Optional |
| `state_demand` | Zenodo (real) | Daily | Peak demand, Temperature, Wind | 📥 Optional |
| **`df_market`** | **Unified merge** | **All** | **All above + `dam_rtm_spread`, `rtm_spike`, `regime`** | **✅ Ready** |

### Next notebook: Phase 0 Feasibility Study
Load `unified_market_dataset.csv` and run the formal statistical tests:
1. Fisher exact test for DAM imbalance → RTM spike association
2. Granger causality from DAM features to RTM prices
3. Multi-horizon (1h, 2h, 4h) predictive power evaluation
4. Feature importance analysis (DAM vs RTM signal retention)